# 17 - T4 Resume Packed Recovery

Versao para Colab T4 com treino retomavel.

Objetivo: permitir treinar em blocos, salvar adapter e progresso no Drive, trocar de conta/sessao e continuar depois.

Caminho correto testado:

`student_pruned -> materializar mlp_ternary_2bit.bin -> aplicar/continuar adapter T4 -> avaliar`

Como usar:

1. Rode todas as celulas.
2. Se existir `student_packed_ternary_recovered_l78_v2` do notebook `15`, ele importa automaticamente esse adapter como progresso de `800` steps.
3. A cada marco (`100`, `200`, `400`, `800`, `1200`) ele salva adapter e JSON.
4. Se a sessao cair, abra de novo e rode tudo. Ele detecta o ultimo marco salvo e continua do proximo.
5. Se quiser parar cedo, pare depois que um marco for salvo.

Arquivos principais no Drive:

- `checkpoints/student_packed_ternary_recovered_l78_t4_resume/`
- `results/packed_recovery/17_t4_resume_progress.json`
- `results/packed_recovery/17_t4_resume_report.json`


In [ ]:
# Auto-bootstrap para Colab/local. Rode esta célula primeiro.
from pathlib import Path
import os, sys, json, gc, re, time, hashlib, zipfile, subprocess, shutil

IN_COLAB = 'google.colab' in sys.modules
PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = '/content/drive/MyDrive/RPT'
LOCAL_ROOT = str(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path(COLAB_DRIVE_ROOT) / PROJECT_NAME
else:
    PROJECT_ROOT = Path(LOCAL_ROOT) / PROJECT_NAME if Path(LOCAL_ROOT).name != PROJECT_NAME else Path(LOCAL_ROOT)

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
CHECKPOINTS_DIR = PROJECT_ROOT / 'checkpoints'
DEPLOY_DIR = PROJECT_ROOT / 'deploy'
EXPORTS_DIR = PROJECT_ROOT / 'exports'
AUDIT_DIR = RESULTS_DIR / 'deployment_audit'
PACKED_RECOVERY_RESULTS_DIR = RESULTS_DIR / 'packed_recovery'
for p in [DATA_DIR, RESULTS_DIR, CHECKPOINTS_DIR, DEPLOY_DIR, EXPORTS_DIR, AUDIT_DIR, PACKED_RECOVERY_RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('CHECKPOINTS_DIR =', CHECKPOINTS_DIR)
print('DEPLOY_DIR =', DEPLOY_DIR)
print('PACKED_RECOVERY_RESULTS_DIR =', PACKED_RECOVERY_RESULTS_DIR)


In [ ]:
# Dependencias.
# torchao nao e necessario aqui; versoes antigas podem quebrar PEFT/LoRA.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-U',
    'transformers',
    'accelerate',
    'sentencepiece',
    'safetensors',
    'peft',
    'datasets',
])


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

HF_TOKEN = os.environ.get('HF_TOKEN')
PRUNED_CHECKPOINT = CHECKPOINTS_DIR / 'student_pruned'
OLD_L78_ADAPTER = CHECKPOINTS_DIR / 'student_ternary_recovered_l78'
PACKED_RECOVERY_OUTPUT = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_t4_resume'
QUESTIONS_PATH = DATA_DIR / 'gemma4_100_questions.json'

QUESTIONS_PAYLOAD = '[\n  {\n    "id": 1,\n    "question": "Qual a capital da França?",\n    "expected_answer": "Paris",\n    "accepted_answers": [\n      "Paris"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 2,\n    "question": "Qual a capital da Alemanha?",\n    "expected_answer": "Berlim",\n    "accepted_answers": [\n      "Berlim",\n      "Berlin"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 3,\n    "question": "Qual a capital da Itália?",\n    "expected_answer": "Roma",\n    "accepted_answers": [\n      "Roma",\n      "Rome"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 4,\n    "question": "Qual a capital da Espanha?",\n    "expected_answer": "Madri",\n    "accepted_answers": [\n      "Madri",\n      "Madrid"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 5,\n    "question": "Qual a capital de Portugal?",\n    "expected_answer": "Lisboa",\n    "accepted_answers": [\n      "Lisboa"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 6,\n    "question": "Qual a capital do Japão?",\n    "expected_answer": "Tóquio",\n    "accepted_answers": [\n      "Tóquio",\n      "Toquio",\n      "Tokyo"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 7,\n    "question": "Qual a capital da China?",\n    "expected_answer": "Pequim",\n    "accepted_answers": [\n      "Pequim",\n      "Beijing",\n      "Pekin"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 8,\n    "question": "Qual a capital da Argentina?",\n    "expected_answer": "Buenos Aires",\n    "accepted_answers": [\n      "Buenos Aires"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 9,\n    "question": "Qual a capital do Canadá?",\n    "expected_answer": "Ottawa",\n    "accepted_answers": [\n      "Ottawa"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 10,\n    "question": "Qual a capital da Austrália?",\n    "expected_answer": "Camberra",\n    "accepted_answers": [\n      "Camberra",\n      "Canberra"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 11,\n    "question": "Quem pintou a Mona Lisa?",\n    "expected_answer": "Leonardo da Vinci",\n    "accepted_answers": [\n      "Leonardo da Vinci",\n      "Da Vinci"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 12,\n    "question": "Quem escreveu Dom Quixote?",\n    "expected_answer": "Miguel de Cervantes",\n    "accepted_answers": [\n      "Miguel de Cervantes",\n      "Cervantes"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 13,\n    "question": "Quem escreveu Hamlet?",\n    "expected_answer": "William Shakespeare",\n    "accepted_answers": [\n      "William Shakespeare",\n      "Shakespeare"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 14,\n    "question": "Quem descobriu a penicilina?",\n    "expected_answer": "Alexander Fleming",\n    "accepted_answers": [\n      "Alexander Fleming",\n      "Fleming"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 15,\n    "question": "Quem desenvolveu a teoria da relatividade?",\n    "expected_answer": "Albert Einstein",\n    "accepted_answers": [\n      "Albert Einstein",\n      "Einstein"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 16,\n    "question": "Qual planeta é conhecido como Planeta Vermelho?",\n    "expected_answer": "Marte",\n    "accepted_answers": [\n      "Marte",\n      "Mars"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 17,\n    "question": "Qual é o maior planeta do Sistema Solar?",\n    "expected_answer": "Júpiter",\n    "accepted_answers": [\n      "Júpiter",\n      "Jupiter"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 18,\n    "question": "Quantos continentes existem tradicionalmente?",\n    "expected_answer": "7",\n    "accepted_answers": [\n      "7",\n      "sete"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 19,\n    "question": "Qual é o oceano mais extenso da Terra?",\n    "expected_answer": "Oceano Pacífico",\n    "accepted_answers": [\n      "Oceano Pacífico",\n      "Pacífico",\n      "Pacific Ocean"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 20,\n    "question": "Qual é o metal cujo símbolo químico é Au?",\n    "expected_answer": "Ouro",\n    "accepted_answers": [\n      "Ouro",\n      "Gold"\n    ],\n    "category": "factual",\n    "notes": ""\n  },\n  {\n    "id": 21,\n    "question": "Qual é o resultado de 12 x 8?",\n    "expected_answer": "96",\n    "accepted_answers": [\n      "96"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 22,\n    "question": "Quanto é 144 dividido por 12?",\n    "expected_answer": "12",\n    "accepted_answers": [\n      "12"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 23,\n    "question": "Quanto é 15 ao quadrado?",\n    "expected_answer": "225",\n    "accepted_answers": [\n      "225"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 24,\n    "question": "Qual é a raiz quadrada de 81?",\n    "expected_answer": "9",\n    "accepted_answers": [\n      "9"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 25,\n    "question": "Quanto é 7 + 18?",\n    "expected_answer": "25",\n    "accepted_answers": [\n      "25"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 26,\n    "question": "Quanto é 100 - 37?",\n    "expected_answer": "63",\n    "accepted_answers": [\n      "63"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 27,\n    "question": "Quanto é 9 x 9?",\n    "expected_answer": "81",\n    "accepted_answers": [\n      "81"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 28,\n    "question": "Quanto é 5! (fatorial de 5)?",\n    "expected_answer": "120",\n    "accepted_answers": [\n      "120"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 29,\n    "question": "Qual é o próximo número na sequência 2, 4, 8, 16, ...?",\n    "expected_answer": "32",\n    "accepted_answers": [\n      "32"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 30,\n    "question": "Se um triângulo tem lados 3, 4 e 5, ele é retângulo?",\n    "expected_answer": "Sim",\n    "accepted_answers": [\n      "Sim",\n      "sim",\n      "Yes"\n    ],\n    "category": "math",\n    "notes": ""\n  },\n  {\n    "id": 31,\n    "question": "A água ferve a aproximadamente quantos graus Celsius ao nível do mar?",\n    "expected_answer": "100",\n    "accepted_answers": [\n      "100",\n      "100°C",\n      "100 graus Celsius"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 32,\n    "question": "Qual gás as plantas absorvem principalmente para fazer fotossíntese?",\n    "expected_answer": "Dióxido de carbono",\n    "accepted_answers": [\n      "Dióxido de carbono",\n      "CO2",\n      "Gás carbônico"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 33,\n    "question": "Qual é a fórmula química da água?",\n    "expected_answer": "H2O",\n    "accepted_answers": [\n      "H2O"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 34,\n    "question": "Qual partícula tem carga elétrica negativa?",\n    "expected_answer": "Elétron",\n    "accepted_answers": [\n      "Elétron",\n      "Eletron",\n      "Electron"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 35,\n    "question": "Qual órgão do corpo humano bombeia sangue?",\n    "expected_answer": "Coração",\n    "accepted_answers": [\n      "Coração",\n      "Coracao"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 36,\n    "question": "Qual é o satélite natural da Terra?",\n    "expected_answer": "Lua",\n    "accepted_answers": [\n      "Lua",\n      "Moon"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 37,\n    "question": "Qual é o processo pelo qual um líquido vira vapor?",\n    "expected_answer": "Evaporação",\n    "accepted_answers": [\n      "Evaporação",\n      "Evaporacao"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 38,\n    "question": "Quantos cromossomos tem um ser humano típico em células somáticas?",\n    "expected_answer": "46",\n    "accepted_answers": [\n      "46"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 39,\n    "question": "Qual é a camada da atmosfera onde vivemos?",\n    "expected_answer": "Troposfera",\n    "accepted_answers": [\n      "Troposfera"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 40,\n    "question": "O DNA tem forma clássica de qual estrutura?",\n    "expected_answer": "Dupla hélice",\n    "accepted_answers": [\n      "Dupla hélice",\n      "Dupla helix",\n      "Double helix"\n    ],\n    "category": "science",\n    "notes": ""\n  },\n  {\n    "id": 41,\n    "question": "Em que ano o homem pisou na Lua pela primeira vez?",\n    "expected_answer": "1969",\n    "accepted_answers": [\n      "1969"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 42,\n    "question": "Quem foi o primeiro presidente do Brasil?",\n    "expected_answer": "Deodoro da Fonseca",\n    "accepted_answers": [\n      "Deodoro da Fonseca"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 43,\n    "question": "Quem foi o líder sul-africano associado ao fim do apartheid e eleito presidente em 1994?",\n    "expected_answer": "Nelson Mandela",\n    "accepted_answers": [\n      "Nelson Mandela",\n      "Mandela"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 44,\n    "question": "Em que país começou a Revolução Industrial?",\n    "expected_answer": "Reino Unido",\n    "accepted_answers": [\n      "Reino Unido",\n      "Inglaterra",\n      "Grã-Bretanha",\n      "Great Britain"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 45,\n    "question": "Qual império construiu o Coliseu?",\n    "expected_answer": "Império Romano",\n    "accepted_answers": [\n      "Império Romano",\n      "Roman Empire",\n      "Roma"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 46,\n    "question": "Em que ano caiu o Muro de Berlim?",\n    "expected_answer": "1989",\n    "accepted_answers": [\n      "1989"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 47,\n    "question": "Quem liderou a independência da Índia com estratégia de não violência?",\n    "expected_answer": "Mahatma Gandhi",\n    "accepted_answers": [\n      "Mahatma Gandhi",\n      "Gandhi"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 48,\n    "question": "Qual civilização antiga construiu Machu Picchu?",\n    "expected_answer": "Inca",\n    "accepted_answers": [\n      "Inca",\n      "Império Inca"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 49,\n    "question": "Quem foi o imperador francês derrotado em Waterloo?",\n    "expected_answer": "Napoleão Bonaparte",\n    "accepted_answers": [\n      "Napoleão Bonaparte",\n      "Napoleon Bonaparte",\n      "Napoleão"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 50,\n    "question": "Em que continente fica o Egito?",\n    "expected_answer": "África",\n    "accepted_answers": [\n      "África",\n      "Africa"\n    ],\n    "category": "history",\n    "notes": ""\n  },\n  {\n    "id": 51,\n    "question": "Se todos os gatos são mamíferos e Mimi é um gato, Mimi é mamífero?",\n    "expected_answer": "Sim",\n    "accepted_answers": [\n      "Sim",\n      "sim",\n      "Yes"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 52,\n    "question": "Se hoje é segunda-feira, que dia será depois de 3 dias?",\n    "expected_answer": "Quinta-feira",\n    "accepted_answers": [\n      "Quinta-feira",\n      "Quinta",\n      "Thursday"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 53,\n    "question": "Qual palavra não pertence ao grupo: maçã, banana, cenoura, uva?",\n    "expected_answer": "Cenoura",\n    "accepted_answers": [\n      "Cenoura"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 54,\n    "question": "Se João é mais alto que Maria e Maria é mais alta que Ana, quem é a mais alta?",\n    "expected_answer": "João",\n    "accepted_answers": [\n      "João",\n      "Joao"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 55,\n    "question": "Um carro percorre 60 km em 1 hora. Em 3 horas, mantendo a velocidade, percorre quantos km?",\n    "expected_answer": "180",\n    "accepted_answers": [\n      "180"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 56,\n    "question": "Se 2 canetas custam 10 reais, quanto custam 5 canetas ao mesmo preço unitário?",\n    "expected_answer": "25",\n    "accepted_answers": [\n      "25",\n      "25 reais",\n      "R$25"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 57,\n    "question": "Qual número é ímpar: 14, 22, 31, 48?",\n    "expected_answer": "31",\n    "accepted_answers": [\n      "31"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 58,\n    "question": "Se A implica B e A é verdadeiro, o que se conclui sobre B?",\n    "expected_answer": "B é verdadeiro",\n    "accepted_answers": [\n      "B é verdadeiro",\n      "verdadeiro",\n      "B true"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 59,\n    "question": "Se uma receita pede 2 ovos para 1 bolo, quantos ovos são necessários para 4 bolos?",\n    "expected_answer": "8",\n    "accepted_answers": [\n      "8"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 60,\n    "question": "Qual alternativa completa melhor: livro está para leitura assim como garfo está para ...?",\n    "expected_answer": "Comer",\n    "accepted_answers": [\n      "Comer",\n      "alimentação",\n      "eat"\n    ],\n    "category": "logic",\n    "notes": ""\n  },\n  {\n    "id": 61,\n    "question": "Em Python, qual palavra-chave define uma função?",\n    "expected_answer": "def",\n    "accepted_answers": [\n      "def"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 62,\n    "question": "Qual estrutura em Python normalmente guarda pares chave-valor?",\n    "expected_answer": "dicionário",\n    "accepted_answers": [\n      "dicionário",\n      "dicionario",\n      "dict"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 63,\n    "question": "Em JavaScript, qual valor booleano representa verdade?",\n    "expected_answer": "true",\n    "accepted_answers": [\n      "true"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 64,\n    "question": "Qual comando Git cria um novo commit com as mudanças staged?",\n    "expected_answer": "git commit",\n    "accepted_answers": [\n      "git commit"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 65,\n    "question": "Qual protocolo é usado normalmente para páginas web seguras?",\n    "expected_answer": "HTTPS",\n    "accepted_answers": [\n      "HTTPS"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 66,\n    "question": "Qual comando Linux lista arquivos de um diretório?",\n    "expected_answer": "ls",\n    "accepted_answers": [\n      "ls"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 67,\n    "question": "Em SQL, qual comando retorna linhas de uma tabela?",\n    "expected_answer": "SELECT",\n    "accepted_answers": [\n      "SELECT"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 68,\n    "question": "Em HTML, qual tag principal representa um link?",\n    "expected_answer": "<a>",\n    "accepted_answers": [\n      "<a>",\n      "a"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 69,\n    "question": "Em CSS, qual propriedade muda a cor do texto?",\n    "expected_answer": "color",\n    "accepted_answers": [\n      "color"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 70,\n    "question": "Qual estrutura de dados usa a política LIFO?",\n    "expected_answer": "pilha",\n    "accepted_answers": [\n      "pilha",\n      "stack"\n    ],\n    "category": "coding",\n    "notes": ""\n  },\n  {\n    "id": 71,\n    "question": "Qual é o idioma oficial mais falado no Brasil?",\n    "expected_answer": "Português",\n    "accepted_answers": [\n      "Português",\n      "Portugues"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 72,\n    "question": "Qual é o antônimo de “rápido”?",\n    "expected_answer": "Lento",\n    "accepted_answers": [\n      "Lento"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 73,\n    "question": "Complete: cachorro está para latir assim como gato está para ...?",\n    "expected_answer": "miar",\n    "accepted_answers": [\n      "miar"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 74,\n    "question": "Qual é o plural de “papel” em português?",\n    "expected_answer": "papéis",\n    "accepted_answers": [\n      "papéis",\n      "papeis"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 75,\n    "question": "Qual palavra é um verbo: azul, correr, mesa, janela?",\n    "expected_answer": "correr",\n    "accepted_answers": [\n      "correr"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 76,\n    "question": "Qual é o sinônimo mais próximo de “feliz”?",\n    "expected_answer": "contente",\n    "accepted_answers": [\n      "contente",\n      "alegre"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 77,\n    "question": "Qual é a forma correta: “a gente fomos” ou “a gente foi”?",\n    "expected_answer": "a gente foi",\n    "accepted_answers": [\n      "a gente foi"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 78,\n    "question": "Em inglês, como se traduz “casa”?",\n    "expected_answer": "house",\n    "accepted_answers": [\n      "house"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 79,\n    "question": "Qual dessas palavras é um adjetivo: bonito, correr, pedra, ontem?",\n    "expected_answer": "bonito",\n    "accepted_answers": [\n      "bonito"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 80,\n    "question": "Qual palavra está escrita corretamente: exceção ou excessão?",\n    "expected_answer": "exceção",\n    "accepted_answers": [\n      "exceção",\n      "excecao"\n    ],\n    "category": "language",\n    "notes": ""\n  },\n  {\n    "id": 81,\n    "question": "Qual país venceu a Copa do Mundo de 2002?",\n    "expected_answer": "Brasil",\n    "accepted_answers": [\n      "Brasil",\n      "Brazil"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 82,\n    "question": "Quantos jogadores cada time tem em campo no futebol tradicional?",\n    "expected_answer": "11",\n    "accepted_answers": [\n      "11",\n      "onze"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 83,\n    "question": "Em que esporte se usa uma cesta e uma bola laranja?",\n    "expected_answer": "basquete",\n    "accepted_answers": [\n      "basquete",\n      "basketball"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 84,\n    "question": "Em qual esporte Rafael Nadal ficou mundialmente famoso?",\n    "expected_answer": "tênis",\n    "accepted_answers": [\n      "tênis",\n      "tenis",\n      "tennis"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 85,\n    "question": "Qual país sediou as Olimpíadas de 2016?",\n    "expected_answer": "Brasil",\n    "accepted_answers": [\n      "Brasil",\n      "Brazil"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 86,\n    "question": "Qual piloto brasileiro é tricampeão mundial de Fórmula 1 e morreu em 1994?",\n    "expected_answer": "Ayrton Senna",\n    "accepted_answers": [\n      "Ayrton Senna",\n      "Senna"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 87,\n    "question": "No xadrez, quantas casas tem um tabuleiro?",\n    "expected_answer": "64",\n    "accepted_answers": [\n      "64"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 88,\n    "question": "Qual esporte usa raquete, rede e peteca?",\n    "expected_answer": "badminton",\n    "accepted_answers": [\n      "badminton"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 89,\n    "question": "Em que esporte se faz touchdown?",\n    "expected_answer": "futebol americano",\n    "accepted_answers": [\n      "futebol americano",\n      "football americano",\n      "american football"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 90,\n    "question": "Qual seleção é conhecida como La Albiceleste?",\n    "expected_answer": "Argentina",\n    "accepted_answers": [\n      "Argentina"\n    ],\n    "category": "sports",\n    "notes": ""\n  },\n  {\n    "id": 91,\n    "question": "Se você misturar azul com amarelo, qual cor obtém normalmente?",\n    "expected_answer": "verde",\n    "accepted_answers": [\n      "verde"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 92,\n    "question": "Qual eletrodoméstico é usado para manter alimentos congelados?",\n    "expected_answer": "freezer",\n    "accepted_answers": [\n      "freezer",\n      "congelador"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 93,\n    "question": "Quantos minutos tem 2 horas?",\n    "expected_answer": "120",\n    "accepted_answers": [\n      "120"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 94,\n    "question": "Qual objeto é usado para cortar papel com as mãos?",\n    "expected_answer": "tesoura",\n    "accepted_answers": [\n      "tesoura"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 95,\n    "question": "Qual é o mês que vem depois de abril?",\n    "expected_answer": "maio",\n    "accepted_answers": [\n      "maio"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 96,\n    "question": "Se um produto custa 50 reais e recebe desconto de 10 reais, quanto passa a custar?",\n    "expected_answer": "40",\n    "accepted_answers": [\n      "40",\n      "40 reais",\n      "R$40"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 97,\n    "question": "Qual bebida é tradicionalmente feita a partir de grãos de café torrados?",\n    "expected_answer": "café",\n    "accepted_answers": [\n      "café",\n      "cafe"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 98,\n    "question": "Qual direção é oposta ao norte?",\n    "expected_answer": "sul",\n    "accepted_answers": [\n      "sul",\n      "south"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 99,\n    "question": "Qual animal normalmente põe ovos e tem penas?",\n    "expected_answer": "ave",\n    "accepted_answers": [\n      "ave",\n      "pássaro",\n      "passaro",\n      "bird"\n    ],\n    "category": "everyday",\n    "notes": ""\n  },\n  {\n    "id": 100,\n    "question": "Quantos dias tem uma semana?",\n    "expected_answer": "7",\n    "accepted_answers": [\n      "7",\n      "sete"\n    ],\n    "category": "everyday",\n    "notes": ""\n  }\n]'
if not QUESTIONS_PATH.exists():
    QUESTIONS_PATH.write_text(QUESTIONS_PAYLOAD, encoding='utf-8')
    print('QUESTIONS_PATH criado:', QUESTIONS_PATH)

PACKAGE_CANDIDATES = [
    DEPLOY_DIR,
    EXPORTS_DIR / 'ternary_mlp_l78',
    PROJECT_ROOT / 'deploy',
    PROJECT_ROOT / 'exports' / 'ternary_mlp_l78',
]
PACKAGE_DIR = None
for candidate in PACKAGE_CANDIDATES:
    if (candidate / 'manifest.json').exists() and (candidate / 'mlp_ternary_2bit.bin').exists():
        PACKAGE_DIR = candidate
        break
if PACKAGE_DIR is None:
    checked = [str(p) for p in PACKAGE_CANDIDATES]
    raise FileNotFoundError('Pacote packed não encontrado em: ' + json.dumps(checked, ensure_ascii=False, indent=2))

MANIFEST_PATH = PACKAGE_DIR / 'manifest.json'
BIN_PATH = PACKAGE_DIR / 'mlp_ternary_2bit.bin'

PROMPT_TEMPLATE = 'Responda de forma curta e direta.\nPergunta: {question}\nResposta:'
MAX_NEW_TOKENS = 24
BATCH_SIZE = 1
TRAIN_PHASES = [
    {'name': 'phase_100', 'total_steps': 100, 'continue_if_accuracy_at_least': 0.66},
    {'name': 'phase_200', 'total_steps': 200, 'continue_if_accuracy_at_least': 0.70},
    {'name': 'phase_400', 'total_steps': 400, 'continue_if_accuracy_at_least': 0.74},
    {'name': 'phase_800', 'total_steps': 800, 'continue_if_accuracy_at_least': 0.80},
    {'name': 'phase_1200', 'total_steps': 1200, 'continue_if_accuracy_at_least': 0.86},
]
TRAIN_STEPS = TRAIN_PHASES[-1]['total_steps']
LEARNING_RATE = 3e-5
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_ACCURACY = 0.90
REFERENCE_CLEAN_BASE_ADAPTER_ACCURACY = 0.92
PACKED_OLD_ADAPTER_ACCURACY = 0.63
T4_BATCH_SIZE = 1
T4_GRAD_ACCUM = 8
KL_WEIGHT = 0.30
CE_WEIGHT = 0.70
DISTILL_TEMPERATURE = 2.0
EARLY_STOP_ENABLED = False
# Em T4, deixe False para salvar em todos os marcos e decidir manualmente depois.
FORCE_CONTINUE_AFTER_LOW_SCORE = True


print('PRUNED_CHECKPOINT', PRUNED_CHECKPOINT.exists(), PRUNED_CHECKPOINT)
print('OLD_L78_ADAPTER', OLD_L78_ADAPTER.exists(), OLD_L78_ADAPTER)
PROGRESS_PATH = PACKED_RECOVERY_RESULTS_DIR / '17_t4_resume_progress.json'
REPORT_PATH = PACKED_RECOVERY_RESULTS_DIR / '17_t4_resume_report.json'
IMPORT_V2_IF_AVAILABLE = True
SOURCE_V2_ADAPTER = CHECKPOINTS_DIR / 'student_packed_ternary_recovered_l78_v2'
IMPORTED_V2_COMPLETED_STEPS = 800
IMPORTED_V2_ACCURACY = 0.68
IMPORT_MARKER_PATH = PACKED_RECOVERY_OUTPUT / 'imported_from_15_v2.json'
print('PACKED_RECOVERY_OUTPUT', PACKED_RECOVERY_OUTPUT)
print('PROGRESS_PATH', PROGRESS_PATH)
print('SOURCE_V2_ADAPTER', SOURCE_V2_ADAPTER)
print('QUESTIONS_PATH', QUESTIONS_PATH.exists(), QUESTIONS_PATH)
print('PACKAGE_DIR', PACKAGE_DIR)
print('MANIFEST', MANIFEST_PATH.exists(), MANIFEST_PATH)
print('BIN', BIN_PATH.exists(), BIN_PATH, BIN_PATH.stat().st_size if BIN_PATH.exists() else None)


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def timestamped_result_path(directory, stem):
    ts = time.strftime('%Y%m%d_%H%M%S')
    return Path(directory) / f'{stem}_{ts}.json'


def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

manifest = load_json(MANIFEST_PATH)
questions = load_json(QUESTIONS_PATH)
assert manifest.get('format') == 'rpt_hybrid_ternary_mlp_lora_v1', manifest.get('format')
assert manifest['summary']['num_tensors'] == 78, manifest['summary']
assert BIN_PATH.stat().st_size == manifest['summary']['packed_total_bytes'], (BIN_PATH.stat().st_size, manifest['summary']['packed_total_bytes'])
print(json.dumps(manifest['summary'], ensure_ascii=False, indent=2))
print('bin_sha256 =', sha256_file(BIN_PATH))


In [ ]:
# 2-bit unpack e materialização dos 78 MLPs.
def unpack_codes_from_blob(blob, n):
    raw = np.frombuffer(blob, dtype=np.uint8)
    codes = np.empty(raw.size * 4, dtype=np.uint8)
    codes[0::4] = raw & 0b11
    codes[1::4] = (raw >> 2) & 0b11
    codes[2::4] = (raw >> 4) & 0b11
    codes[3::4] = (raw >> 6) & 0b11
    return codes[:n]


def unpack_tensor_from_blob(blob, n, shape, scale, dtype=torch.float32):
    codes = unpack_codes_from_blob(blob, n)
    vals = np.zeros(n, dtype=np.float32)
    vals[codes == 1] = scale
    vals[codes == 2] = -scale
    tensor = torch.from_numpy(vals).view(tuple(shape))
    return tensor.to(dtype=dtype)


def materialize_packed_mlp_into_model(model, manifest, bin_path):
    named_params = dict(model.named_parameters())
    missing_params = []
    meta_params = []
    applied = []
    with open(bin_path, 'rb') as f:
        for item in manifest['tensors']:
            name = item['name']
            param = named_params.get(name)
            if param is None:
                missing_params.append(name)
                continue
            if getattr(param, 'is_meta', False):
                meta_params.append(name)
                continue
            f.seek(item['offset'])
            blob = f.read(item['packed_bytes'])
            recon = unpack_tensor_from_blob(blob, item['numel'], item['shape'], item['scale'], dtype=torch.float32)
            if tuple(param.shape) != tuple(recon.shape):
                raise RuntimeError(f'Shape mismatch for {name}: param={tuple(param.shape)} recon={tuple(recon.shape)}')
            with torch.no_grad():
                param.copy_(recon.to(device=param.device, dtype=param.dtype))
            applied.append(name)
            del recon
    if missing_params or meta_params:
        raise RuntimeError(json.dumps({'missing_params': missing_params, 'meta_params': meta_params}, ensure_ascii=False, indent=2))
    if len(applied) != 78:
        raise RuntimeError(f'Expected 78 MLP tensors applied, got {len(applied)}')
    return applied


def load_pruned_model(device_map='auto'):
    model = AutoModelForCausalLM.from_pretrained(
        str(PRUNED_CHECKPOINT),
        dtype='auto',
        device_map=device_map,
        token=HF_TOKEN,
    )
    model.eval()
    return model


def load_packed_base_model(device_map='auto'):
    model = load_pruned_model(device_map=device_map)
    applied = materialize_packed_mlp_into_model(model, manifest, BIN_PATH)
    model.eval()
    return model, applied


In [ ]:
# Avaliação local alinhada aos notebooks anteriores, usando accepted_answers.
def normalize_text(s):
    return re.sub(r'\s+', ' ', str(s).strip().lower())


def judge_answer(question_payload, answer):
    a = normalize_text(answer)
    accepted = [question_payload.get('expected_answer', '')]
    accepted.extend(question_payload.get('accepted_answers') or [])
    for candidate in accepted:
        e = normalize_text(candidate)
        if e and (e in a or a in e):
            return True
    return False


def extract_answer(prompt, generated):
    text = generated
    if prompt in text:
        text = text.split(prompt, 1)[-1]
    text = text.strip()
    text = text.split('\n')[0].strip()
    return text


def run_eval(model, tokenizer, questions, batch_size=1, max_new_tokens=24, verbose=True):
    rows = []
    correct = 0
    device = next(model.parameters()).device
    for idx, q in enumerate(questions):
        prompt = PROMPT_TEMPLATE.format(question=q['question'])
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(out[0], skip_special_tokens=True)
        answer = extract_answer(prompt, generated)
        ok = judge_answer(q, answer)
        correct += int(ok)
        row = {
            'id': q.get('id', idx + 1),
            'category': q.get('category'),
            'question': q['question'],
            'expected_answer': q['expected_answer'],
            'model_answer': answer,
            'is_correct': ok,
            'raw_output': generated,
        }
        rows.append(row)
        if verbose:
            print(f"{idx+1:03d}/{len(questions)}", 'OK' if ok else 'ERR', '|', q['question'], '=>', answer)
    by_category = {}
    for row in rows:
        cat = row.get('category') or 'unknown'
        by_category.setdefault(cat, {'total': 0, 'correct': 0})
        by_category[cat]['total'] += 1
        by_category[cat]['correct'] += int(row['is_correct'])
    for cat, vals in by_category.items():
        vals['accuracy'] = vals['correct'] / vals['total'] if vals['total'] else 0.0
    summary = {
        'total_questions': len(questions),
        'correct': correct,
        'accuracy': correct / len(questions) if questions else 0.0,
        'by_category': by_category,
    }
    return rows, summary


In [ ]:
# Tokenizacao answer-only e trainer de recovery com professor limpo.
tokenizer = AutoTokenizer.from_pretrained(str(PRUNED_CHECKPOINT), token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_TRAIN_LENGTH = 160

def build_answer_only_features(q):
    prompt = PROMPT_TEMPLATE.format(question=q['question']) + ' '
    answer = str(q['expected_answer']).strip() + tokenizer.eos_token
    prompt_ids = tokenizer(prompt, add_special_tokens=True)['input_ids']
    answer_ids = tokenizer(answer, add_special_tokens=False)['input_ids']
    input_ids = (prompt_ids + answer_ids)[:MAX_TRAIN_LENGTH]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_TRAIN_LENGTH]
    attention_mask = [1] * len(input_ids)
    pad_len = MAX_TRAIN_LENGTH - len(input_ids)
    if pad_len > 0:
        input_ids += [tokenizer.pad_token_id] * pad_len
        labels += [-100] * pad_len
        attention_mask += [0] * pad_len
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

train_rows = []
for q in questions:
    row = build_answer_only_features(q)
    row['question_id'] = q.get('id')
    train_rows.append(row)
train_ds = Dataset.from_list(train_rows)

class PackedRecoveryTrainer(Trainer):
    def __init__(self, teacher_model=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits
        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs.get('attention_mask'),
            )
            teacher_logits = teacher_outputs.logits

        ce_loss = student_outputs.loss
        t = DISTILL_TEMPERATURE
        # Alinha o KL com o shift causal: logits[:, t] predizem labels[:, t+1].
        student_shift = student_logits[:, :-1, :]
        teacher_shift = teacher_logits[:, :-1, :]
        labels_shift = labels[:, 1:]
        token_kl = torch.nn.functional.kl_div(
            torch.nn.functional.log_softmax(student_shift / t, dim=-1),
            torch.nn.functional.softmax(teacher_shift / t, dim=-1),
            reduction='none',
        ).sum(dim=-1) * (t * t)
        mask = (labels_shift != -100).to(token_kl.dtype)
        kl_loss = (token_kl * mask).sum() / mask.sum().clamp_min(1.0)
        loss = CE_WEIGHT * ce_loss + KL_WEIGHT * kl_loss
        return (loss, student_outputs) if return_outputs else loss

class SimpleCausalCollator:
    def __call__(self, features):
        keys = ['input_ids', 'attention_mask', 'labels']
        return {k: torch.tensor([f[k] for f in features], dtype=torch.long) for k in keys}

print('train examples:', len(train_ds))
print('answer-only labels active tokens example:', sum(x != -100 for x in train_ds[0]['labels']))


In [ ]:
# Importa automaticamente o adapter v2 do notebook 15, se existir.
# Isso permite continuar de 800 steps sem copiar/criar progress manualmente.
progress_exists = PROGRESS_PATH.exists()
dst_has_adapter = (PACKED_RECOVERY_OUTPUT / 'adapter_model.safetensors').exists()
src_has_adapter = (SOURCE_V2_ADAPTER / 'adapter_model.safetensors').exists()

if IMPORT_V2_IF_AVAILABLE and (not progress_exists) and (not dst_has_adapter) and src_has_adapter:
    print('Importando adapter do 15 para caminho retomavel:')
    print('  src =', SOURCE_V2_ADAPTER)
    print('  dst =', PACKED_RECOVERY_OUTPUT)
    if PACKED_RECOVERY_OUTPUT.exists():
        raise RuntimeError(f'Destino existe mas nao tem adapter_model.safetensors: {PACKED_RECOVERY_OUTPUT}')
    shutil.copytree(SOURCE_V2_ADAPTER, PACKED_RECOVERY_OUTPUT)
    save_json({
        'imported_from': str(SOURCE_V2_ADAPTER),
        'imported_to': str(PACKED_RECOVERY_OUTPUT),
        'assumed_completed_steps': IMPORTED_V2_COMPLETED_STEPS,
        'assumed_accuracy': IMPORTED_V2_ACCURACY,
        'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'note': 'Imported automatically by 17_t4_resume from notebook 15 v2 adapter.',
    }, IMPORT_MARKER_PATH)
    dst_has_adapter = True

if IMPORT_V2_IF_AVAILABLE and (not PROGRESS_PATH.exists()) and dst_has_adapter and IMPORT_MARKER_PATH.exists():
    marker = load_json(IMPORT_MARKER_PATH)
    progress = {
        'completed_steps': int(marker.get('assumed_completed_steps', IMPORTED_V2_COMPLETED_STEPS)),
        'best_accuracy': float(marker.get('assumed_accuracy', IMPORTED_V2_ACCURACY)),
        'best_phase': 'imported_from_15_v2_800',
        'adapter_path': str(PACKED_RECOVERY_OUTPUT),
        'phase_results': [],
        'baselines': {
            'packed_base_no_adapter': {'summary': {'accuracy': 0.01}, 'note': 'Historical reference from packed path tests.'},
            'old_l78_on_packed': {'summary': {'accuracy': PACKED_OLD_ADAPTER_ACCURACY}, 'note': 'Historical reference from notebook 14/15.'},
        },
        'imported_from': marker.get('imported_from'),
        'imported_at': marker.get('created_at'),
        'note': 'Imported from notebook 15 packed recovery v2. Accuracy assumed from 15 result: 0.68.',
    }
    save_json(progress, PROGRESS_PATH)
    print('Progress importado salvo em:', PROGRESS_PATH)
    print(json.dumps({k: progress[k] for k in ['completed_steps', 'best_accuracy', 'best_phase', 'adapter_path']}, ensure_ascii=False, indent=2))
elif PROGRESS_PATH.exists():
    existing = load_json(PROGRESS_PATH)
    print('Progress existente:', PROGRESS_PATH)
    print(json.dumps({
        'completed_steps': existing.get('completed_steps'),
        'best_accuracy': existing.get('best_accuracy'),
        'best_phase': existing.get('best_phase'),
        'adapter_path': existing.get('adapter_path'),
    }, ensure_ascii=False, indent=2))
elif dst_has_adapter:
    print('AVISO: adapter retomavel existe, mas nao ha progress JSON nem marker de importacao.')
    print('Se esse adapter veio do 15, crie/remova o marker ou apague o destino para importar automaticamente.')
else:
    print('Nenhum adapter previo importado. O treino comecara do zero.')


In [ ]:
# Baselines rapidos. Se ja existir progresso, pula para economizar tempo.
cleanup_memory()
phase_results = []
progress = load_json(PROGRESS_PATH) if PROGRESS_PATH.exists() else None

packed_pure_eval = None
old_adapter_eval = None

if progress:
    print('Progresso existente detectado; pulando baselines para preservar resume.')
    phase_results = progress.get('phase_results', [])
    baselines = progress.get('baselines', {})
else:
    packed_model, applied = load_packed_base_model(device_map='auto')
    packed_model.eval()
    rows, summary = run_eval(packed_model, tokenizer, questions, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS, verbose=False)
    packed_pure_eval = {
        'label': 'packed_base_no_adapter',
        'applied_packed_mlp_tensors': len(applied),
        'summary': summary,
        'rows': rows,
    }
    out_path = timestamped_result_path(PACKED_RECOVERY_RESULTS_DIR, '17_t4_packed_base_no_adapter_eval')
    save_json(packed_pure_eval, out_path)
    print('Packed base sem adapter:', summary, '| salvo em:', out_path)
    del packed_model
    cleanup_memory()

    if OLD_L78_ADAPTER.exists():
        packed_model, applied = load_packed_base_model(device_map='auto')
        old_adapter_model = PeftModel.from_pretrained(packed_model, str(OLD_L78_ADAPTER), token=HF_TOKEN, is_trainable=False)
        old_adapter_model.eval()
        rows, summary = run_eval(old_adapter_model, tokenizer, questions, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS, verbose=False)
        old_adapter_eval = {
            'label': 'old_l78_adapter_on_packed',
            'adapter': str(OLD_L78_ADAPTER),
            'applied_packed_mlp_tensors': len(applied),
            'summary': summary,
            'rows': rows,
        }
        out_path = timestamped_result_path(PACKED_RECOVERY_RESULTS_DIR, '17_t4_old_l78_on_packed_eval')
        save_json(old_adapter_eval, out_path)
        print('Old L78 em packed:', summary, '| salvo em:', out_path)
        del old_adapter_model, packed_model
        cleanup_memory()
    else:
        print('OLD_L78_ADAPTER ausente; pulando baseline antigo.')

    save_json({
        'completed_steps': 0,
        'best_accuracy': -1.0,
        'best_phase': None,
        'phase_results': [],
        'baselines': {
            'packed_base_no_adapter': packed_pure_eval,
            'old_l78_on_packed': old_adapter_eval,
        },
    }, PROGRESS_PATH)
    print('Progresso inicial salvo em:', PROGRESS_PATH)


In [ ]:
# Treino T4 retomavel. Rode novamente em outra sessao para continuar do ultimo marco salvo.
cleanup_memory()

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('AVISO: sem CUDA. Este notebook foi planejado para T4.')

progress = load_json(PROGRESS_PATH) if PROGRESS_PATH.exists() else {
    'completed_steps': 0,
    'best_accuracy': -1.0,
    'best_phase': None,
    'phase_results': [],
    'baselines': {},
}
completed_steps = int(progress.get('completed_steps', 0))
best_accuracy = float(progress.get('best_accuracy', -1.0))
best_phase = progress.get('best_phase')
phase_results = progress.get('phase_results', [])
baselines = progress.get('baselines', {})

print('Retomando de completed_steps =', completed_steps, 'best_accuracy =', best_accuracy, 'best_phase =', best_phase)

teacher_model = load_pruned_model(device_map='auto')
teacher_model.eval()
teacher_model.config.use_cache = False

student_model, applied = load_packed_base_model(device_map='auto')
student_model.eval()
student_model.config.use_cache = False
if hasattr(student_model, 'gradient_checkpointing_enable'):
    student_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)

if PACKED_RECOVERY_OUTPUT.exists() and (PACKED_RECOVERY_OUTPUT / 'adapter_model.safetensors').exists():
    print('Carregando adapter existente:', PACKED_RECOVERY_OUTPUT)
    student_model = PeftModel.from_pretrained(student_model, str(PACKED_RECOVERY_OUTPUT), token=HF_TOKEN, is_trainable=True)
else:
    print('Criando adapter novo:', PACKED_RECOVERY_OUTPUT)
    student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()

start_wall = time.time()

for phase in TRAIN_PHASES:
    target_total = int(phase['total_steps'])
    additional_steps = target_total - completed_steps
    if additional_steps <= 0:
        print('Pulando fase ja concluida:', phase['name'], 'target_total=', target_total)
        continue

    print('\n' + '=' * 100)
    print('Fase', phase['name'], '| additional_steps=', additional_steps, '| target_total=', target_total)

    args = TrainingArguments(
        output_dir=str(PACKED_RECOVERY_OUTPUT),
        per_device_train_batch_size=T4_BATCH_SIZE,
        gradient_accumulation_steps=T4_GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        max_steps=additional_steps,
        logging_steps=10,
        save_strategy='no',
        bf16=torch.cuda.is_available(),
        report_to=[],
        remove_unused_columns=False,
        dataloader_num_workers=0,
    )

    trainer = PackedRecoveryTrainer(
        model=student_model,
        teacher_model=teacher_model,
        args=args,
        train_dataset=train_ds,
        data_collator=SimpleCausalCollator(),
    )
    train_output = trainer.train()
    completed_steps = target_total

    student_model.eval()
    rows, summary = run_eval(student_model, tokenizer, questions, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS, verbose=False)
    phase_eval = {
        'phase': phase['name'],
        'total_steps': completed_steps,
        'additional_steps': additional_steps,
        'summary': summary,
        'rows': rows,
        'train_metrics': train_output.metrics,
        'elapsed_minutes_this_session': round((time.time() - start_wall) / 60, 3),
        'comparison': {
            'packed_base_no_adapter_accuracy': (baselines.get('packed_base_no_adapter') or {}).get('summary', {}).get('accuracy'),
            'old_l78_on_packed_accuracy': (baselines.get('old_l78_on_packed') or {}).get('summary', {}).get('accuracy'),
            'clean_base_plus_l78_reference_accuracy': REFERENCE_CLEAN_BASE_ADAPTER_ACCURACY,
            'delta_vs_old_l78_on_packed': None if not (baselines.get('old_l78_on_packed') or {}).get('summary') else round(summary['accuracy'] - baselines['old_l78_on_packed']['summary']['accuracy'], 6),
            'delta_vs_clean_base_l78_reference': round(summary['accuracy'] - REFERENCE_CLEAN_BASE_ADAPTER_ACCURACY, 6),
        },
    }
    phase_results.append(phase_eval)
    out_path = timestamped_result_path(PACKED_RECOVERY_RESULTS_DIR, f"17_t4_resume_{phase['name']}")
    save_json(phase_eval, out_path)
    print('Eval fase salvo em:', out_path)
    print(json.dumps({
        'phase': phase['name'],
        'total_steps': completed_steps,
        'accuracy': summary['accuracy'],
        'comparison': phase_eval['comparison'],
        'elapsed_minutes_this_session': phase_eval['elapsed_minutes_this_session'],
    }, ensure_ascii=False, indent=2))

    # Salva sempre no fim de cada marco, mesmo se nao for o melhor, para permitir retomar exatamente dali.
    trainer.save_model(str(PACKED_RECOVERY_OUTPUT))
    tokenizer.save_pretrained(str(PACKED_RECOVERY_OUTPUT))

    if summary['accuracy'] > best_accuracy:
        best_accuracy = summary['accuracy']
        best_phase = phase['name']

    save_json({
        'completed_steps': completed_steps,
        'best_accuracy': best_accuracy,
        'best_phase': best_phase,
        'adapter_path': str(PACKED_RECOVERY_OUTPUT),
        'phase_results': phase_results,
        'baselines': baselines,
        'load_order': 'load student_pruned -> materialize mlp_ternary_2bit.bin -> apply this adapter',
        'packed_manifest': str(MANIFEST_PATH),
        'packed_bin': str(BIN_PATH),
        'packed_bin_sha256': sha256_file(BIN_PATH),
        'lora': {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT},
        'training': {
            'learning_rate': LEARNING_RATE,
            'batch_size': T4_BATCH_SIZE,
            'gradient_accumulation_steps': T4_GRAD_ACCUM,
            'ce_weight': CE_WEIGHT,
            'kl_weight': KL_WEIGHT,
            'temperature': DISTILL_TEMPERATURE,
        },
    }, PROGRESS_PATH)
    save_json(load_json(PROGRESS_PATH), PACKED_RECOVERY_OUTPUT / 'packed_base_required.json')
    print('Adapter/progresso salvos. Pode parar a sessao com seguranca depois desta linha.')

    threshold = phase.get('continue_if_accuracy_at_least')
    if EARLY_STOP_ENABLED and threshold is not None and summary['accuracy'] < threshold:
        print(f"Parando cedo: accuracy {summary['accuracy']:.3f} < threshold {threshold:.3f} da fase {phase['name']}.")
        break

    student_model.train()

print('Fim desta execucao. completed_steps:', completed_steps, 'best_phase:', best_phase, 'best_accuracy:', best_accuracy)


In [ ]:
# Relatorio final e zip leve.
progress = load_json(PROGRESS_PATH)
phase_results = progress.get('phase_results', [])
final_eval = phase_results[-1] if phase_results else None
final_accuracy = final_eval['summary']['accuracy'] if final_eval else None

final_report = {
    'status': 'target_met' if (final_accuracy is not None and final_accuracy >= TARGET_ACCURACY) else 'in_progress_or_target_not_met',
    'notebook': '17_t4_resume_packed_recovery',
    'purpose': 'T4 resumable recovery for packed/materialized ternary MLP path.',
    'adapter_path': str(PACKED_RECOVERY_OUTPUT),
    'completed_steps': progress.get('completed_steps'),
    'best_phase': progress.get('best_phase'),
    'best_accuracy': progress.get('best_accuracy'),
    'final_accuracy': final_accuracy,
    'target_accuracy': TARGET_ACCURACY,
    'phases': [
        {
            'phase': r['phase'],
            'total_steps': r['total_steps'],
            'accuracy': r['summary']['accuracy'],
            'comparison': r['comparison'],
        }
        for r in phase_results
    ],
    'baselines': {
        'packed_base_no_adapter_accuracy': ((progress.get('baselines', {}).get('packed_base_no_adapter') or {}).get('summary') or {}).get('accuracy'),
        'old_l78_on_packed_accuracy': ((progress.get('baselines', {}).get('old_l78_on_packed') or {}).get('summary') or {}).get('accuracy'),
        'clean_base_plus_l78_reference_accuracy': REFERENCE_CLEAN_BASE_ADAPTER_ACCURACY,
        'student_pruned_reference_accuracy': 0.92,
        'hf_gemma_2_2b_it_original_reference_accuracy': 0.90,
    },
    'method': {
        'resumable': True,
        'answer_only_labels': True,
        'ce_weight': CE_WEIGHT,
        'kl_weight': KL_WEIGHT,
        'temperature': DISTILL_TEMPERATURE,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'batch_size': T4_BATCH_SIZE,
        'gradient_accumulation_steps': T4_GRAD_ACCUM,
        'train_phases': TRAIN_PHASES,
    },
    'required_load_order': 'load student_pruned -> materialize mlp_ternary_2bit.bin using manifest.json -> apply this adapter with PeftModel.from_pretrained',
    'packed_bin_sha256': sha256_file(BIN_PATH),
    'manifest_path': str(MANIFEST_PATH),
    'bin_path': str(BIN_PATH),
}
save_json(final_report, REPORT_PATH)
print('REPORT:', REPORT_PATH)
print(json.dumps(final_report, ensure_ascii=False, indent=2)[:12000])

artifact_paths = [REPORT_PATH, PROGRESS_PATH, PACKED_RECOVERY_OUTPUT / 'packed_base_required.json']
for p in sorted(PACKED_RECOVERY_RESULTS_DIR.glob('17_t4_*_eval_*.json')):
    artifact_paths.append(p)
for p in sorted(PACKED_RECOVERY_RESULTS_DIR.glob('17_t4_resume_phase_*.json')):
    artifact_paths.append(p)

zip_path = PACKED_RECOVERY_RESULTS_DIR / '17_t4_resume_packed_recovery_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    seen = set()
    for path in artifact_paths:
        if path.exists() and path not in seen:
            zf.write(path, arcname=path.name)
            seen.add(path)
print('ZIP salvo em:', zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
